# 🚀 Nexus Audio - Treino TPU v2

**Com feedback em tempo real!** Agora você vê cada step.

In [ ]:
# ============================================
# 1. SETUP TPU
# ============================================
import os
import sys

# Forçar output sem buffer
os.environ['PYTHONUNBUFFERED'] = '1'

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    USE_TPU = True
    print("✅ TPU detectada!", flush=True)
except ImportError:
    USE_TPU = False
    print("⚠️ TPU não disponível", flush=True)

!pip install -q einops encodec

In [ ]:
# ============================================
# 2. IMPORTS
# ============================================
import torch
import torch.nn as nn
from pathlib import Path
import time

print(f"PyTorch: {torch.__version__}", flush=True)

if USE_TPU:
    device = xm.xla_device()
    print(f"🚀 TPU: {device}", flush=True)
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}", flush=True)
else:
    device = torch.device('cpu')
    print("💻 CPU", flush=True)

In [ ]:
# ============================================
# 3. CARREGAR CÓDIGO
# ============================================
sys.path.insert(0, "/kaggle/input/nexus-audio-code")
from src.model import SiMBATherapeutic
print("✅ Código carregado!", flush=True)

In [ ]:
# ============================================
# 4. TOKENS
# ============================================
TOKENS_PATH = "/kaggle/input/nexus-fma-tokens"

token_files = list(Path(TOKENS_PATH).glob("**/*.pt"))
print(f"✅ Tokens: {len(token_files)}", flush=True)

if token_files:
    sample = torch.load(token_files[0])
    print(f"Shape: {sample.shape}", flush=True)

In [ ]:
# ============================================
# 5. CONFIG
# ============================================
CONFIG = {
    "model": {
        "d_model": 384,
        "n_layers": 6,
        "d_state": 16,
        "d_conv": 4,
        "expand": 2,
        "max_seq_len": 4096,
    },
    "audio": {
        "sample_rate": 24000,
        "encodec_bandwidth": 6.0,
    },
    "therapeutic": {"use_biofeedback": False},
}

BATCH_SIZE = 8  # Menor pra ser mais estável
LEARNING_RATE = 3e-4
MAX_STEPS = 5000
ACCUM_STEPS = 4
LOG_EVERY = 10  # Log a cada 10 steps!
SAVE_EVERY = 500

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("✅ Config OK", flush=True)

In [ ]:
# ============================================
# 6. MODELO
# ============================================
print("Criando modelo...", flush=True)
model = SiMBATherapeutic.from_config(CONFIG)
model = model.to(device)
print(f"✅ Parâmetros: {sum(p.numel() for p in model.parameters()):,}", flush=True)

In [ ]:
# ============================================
# 7. DATASET
# ============================================
from torch.utils.data import Dataset, DataLoader

class TokenDataset(Dataset):
    def __init__(self, token_dir):
        self.files = list(Path(token_dir).glob("**/*.pt"))
        print(f"Dataset: {len(self.files)} tokens", flush=True)
        
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        tokens = torch.load(self.files[idx])
        if tokens.dim() == 3:
            tokens = tokens.squeeze(0)
        return tokens.long()

dataset = TokenDataset(TOKENS_PATH)

# num_workers=0 evita problemas no TPU!
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # IMPORTANTE!
    drop_last=True,
)
print(f"✅ Batches por época: {len(loader)}", flush=True)

In [ ]:
# ============================================
# 8. TREINO COM LOGS!
# ============================================
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def log(msg):
    """Print com timestamp e flush"""
    timestamp = time.strftime('%H:%M:%S')
    print(f"[{timestamp}] {msg}", flush=True)

def train(model, loader, max_steps, lr, accum=4):
    log("="*50)
    log(f"🚀 INICIANDO TREINO")
    log(f"Device: {'TPU' if USE_TPU else 'GPU/CPU'}")
    log(f"Steps: {max_steps} | Batch: {BATCH_SIZE} | Accum: {accum}")
    log("="*50)
    
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler = CosineAnnealingLR(optimizer, T_max=max_steps, eta_min=lr*0.1)
    
    model.train()
    global_step = 0
    running_loss = 0
    batch_count = 0
    start = time.time()
    epoch = 0
    
    # Checkpoint inicial
    torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/step0.pt")
    log("💾 Checkpoint inicial salvo")
    
    while global_step < max_steps:
        epoch += 1
        log(f"📍 Época {epoch} iniciando...")
        
        for batch_idx, tokens in enumerate(loader):
            tokens = tokens.to(device)
            
            # Forward
            outputs = model(tokens=tokens, labels=tokens)
            loss = outputs["loss"]
            if isinstance(loss, tuple):
                loss = loss[0]
            loss = loss.mean() / accum
            
            # Backward
            loss.backward()
            running_loss += loss.item()
            batch_count += 1
            
            if batch_count % accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                
                if USE_TPU:
                    xm.optimizer_step(optimizer)
                    xm.mark_step()  # Força execução!
                else:
                    optimizer.step()
                
                scheduler.step()
                optimizer.zero_grad()
                
                global_step += 1
                
                # LOG A CADA N STEPS!
                if global_step % LOG_EVERY == 0:
                    avg_loss = running_loss / LOG_EVERY
                    elapsed = (time.time() - start) / 60
                    steps_per_min = global_step / elapsed if elapsed > 0 else 0
                    eta = (max_steps - global_step) / steps_per_min if steps_per_min > 0 else 0
                    
                    log(f"Step {global_step}/{max_steps} | Loss: {avg_loss:.4f} | "
                        f"{steps_per_min:.1f} steps/min | ETA: {eta:.0f}min")
                    running_loss = 0
                
                # Salvar checkpoint
                if global_step % SAVE_EVERY == 0:
                    torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/step{global_step}.pt")
                    log(f"💾 Checkpoint salvo: step{global_step}.pt")
                
                if global_step >= max_steps:
                    break
        
        if global_step >= max_steps:
            break
    
    elapsed = (time.time() - start) / 60
    log("="*50)
    log(f"✅ TREINO COMPLETO!")
    log(f"Tempo total: {elapsed:.1f} min")
    log(f"Steps: {global_step}")
    log("="*50)
    
    torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/final.pt")
    return model

In [ ]:
# ============================================
# 9. TREINAR!
# ============================================
model = train(
    model=model,
    loader=loader,
    max_steps=MAX_STEPS,
    lr=LEARNING_RATE,
    accum=ACCUM_STEPS,
)

In [ ]:
# ============================================
# 10. SALVAR
# ============================================
final = {
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
}
torch.save(final, f"{CHECKPOINT_DIR}/nexus_trained.pt")

!cp /kaggle/working/checkpoints/*.pt /kaggle/working/
!ls -la /kaggle/working/*.pt
print("\n✅ Modelo salvo!", flush=True)

---
# 🎵 Testar

In [ ]:
import torchaudio

model_cpu = model.to('cpu')
model_cpu.eval()

print("Gerando 10s...", flush=True)
with torch.no_grad():
    waveform = model_cpu.generate(duration_seconds=10.0, temperature=0.9)

if waveform.dim() == 3:
    waveform = waveform.squeeze(0)
torchaudio.save("/kaggle/working/output.wav", waveform.cpu(), 24000)
print("✅ Salvo!", flush=True)

In [ ]:
from IPython.display import Audio
Audio("/kaggle/working/output.wav")